# 06 — CNN–Transformer (Reduced-Electrode)

**Family:** CNN–Transformer &nbsp;•&nbsp; **Input:** raw signal, **5 channels only**

Same CNN + Transformer family as notebook 04, but restricted to a consumer-grade
5-electrode montage: **AF3, F3, F4, AF4, O1** — the frontal-asymmetry quartet
(a valence / arousal correlate) plus one occipital reference. This quantifies how
much accuracy is lost when going from a 14-channel research headset to a minimal
wearable layout.

## 1. Setup

In [ ]:
import os, sys

# make the `src` package importable when the notebook runs from notebooks/
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from src import benchmark_si as B
from src.evaluation import set_seed, get_device, count_parameters
from src.data_loader import load_subject, REDUCED_5_CHANNELS
from src.models import cnn_transformer_reduced

set_seed(42)
DEVICE = get_device()
PROCESSED_DIR = os.path.join(ROOT, "data", "processed")

# ----- experiment configuration -----
MODEL_NAME = "CNN_Transformer_Reduced"
LABEL = "valence"     # main comparison target. Swap to "arousal" / "dominance" to reproduce.
QUICK = False         # True  -> 3-fold / few-epoch smoke test
                      # False -> the full 23-fold benchmark used for the thesis table
print("device:", DEVICE, "| model:", MODEL_NAME, "| label:", LABEL, "| QUICK:", QUICK)
print("reduced montage:", REDUCED_5_CHANNELS)
print("recipe:", B.REGISTRY[MODEL_NAME])


## 2. Architecture

`src/models/cnn_transformer_reduced.py`: a 3-block 1-D temporal CNN stem
(downsamples time ×8) → sinusoidal positional encoding → 3 Transformer encoder
layers → additive temporal-attention pooling → linear head. Identical in spirit to
CIT-EmotionNet's global branch, adapted to raw multi-channel time series.

In [ ]:
# Instantiate once on a dummy spec just to print the graph + parameter count.
_dummy_meta = {"input_shape": (5, load_subject(PROCESSED_DIR, "subject_01").window_samples)}
_m = cnn_transformer_reduced.build({"n_classes": 2, "meta": _dummy_meta})
print(_m)
print(f"\ntrainable parameters: {count_parameters(_m):,}")
del _m

## Validation protocol — subject-independent (LOSO)

The benchmark uses a single protocol so every architecture is scored identically:
**23-fold leave-one-subject-out** (`data/splits/loso_splits.json`). Fold *k* trains a
fresh model on 22 subjects and tests on the held-out 23rd; normalisation statistics
are fit on the training subjects only and a 10% slice of them is used for early
stopping. We report the **mean ± std of the headline metrics across the 23 folds**.

The 5-channel montage (`src.data_loader.REDUCED_5_CHANNELS`) is wired into this
model's recipe in `benchmark_si.REGISTRY`. Everything runs through
`src.benchmark_si.run_one`, which trains via `src.evaluation.run_loso` and writes
`results/{MODEL_NAME}_results.json`.


## 3. Run the subject-independent (LOSO) benchmark

In [ ]:
# One shared call: 23-fold LOSO for this model, then writes results/CNN_Transformer_Reduced_results.json
res = B.run_one(MODEL_NAME, label=LABEL, quick=QUICK, device=DEVICE)
res

## Notes for the methodology chapter

* The 5-channel subset is `src/data_loader.REDUCED_5_CHANNELS` and is referenced by
  this model's entry in `src/benchmark_si.py::REGISTRY`; change it in one place to
  test other minimal montages.
* Compare this row directly against notebook 04 (CIT-EmotionNet, 14ch DE) and
  notebook 01 (EEGNet, 14ch raw) for the "how few electrodes can we get away with"
  discussion.
* Every metric comes from `src/evaluation.compute_metrics`; `params_count` and
  `train_time_sec` are logged to `results/{MODEL_NAME}_results.json`.
* Set `QUICK = True` to smoke-test (3 folds / few epochs); `LABEL` switches the
  target between valence / arousal / dominance with no other change.
